# 15 — LASSO Feature Selection

Screens the country-year feature panel from Notebook 14 using L1-regularised logistic
regression (LASSO), then rescues non-linearly associated features that LASSO would
silently discard.

## Method

1. **LASSO screening** — `LogisticRegressionCV(penalty='l1', solver='saga')` with 5
   expanding temporal folds inside the training window (2000–2018). λ is chosen by
   the 1-SE rule (most regularised λ within 1 standard error of best AUPRC).
2. **Mutual information rescue** — `mutual_info_classif` detects features with
   U-shaped, threshold, or interaction-only effects that receive a zero LASSO
   coefficient despite having real predictive information. Features in the MI top-50
   that were zeroed by LASSO and show a large MI-vs-LASSO rank gap are rescued.
3. **Audit table** — per feature: LASSO coefficient, MI score, LASSO rank, MI rank,
   rank gap, rescued flag. Written to ADLS for use in Notebook 17.

## Outputs per outcome
- `feature_selection/{RUN_DATE}/selected_{outcome}.json` — feature manifest
- `feature_selection/{RUN_DATE}/audit_{outcome}.parquet` — audit table
- MLflow: regularisation path plot (λ vs. coefficient magnitude)

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER       (default: 'data')
AZUREML_MLFLOW_URI   (optional, for MLflow tracking)
```

In [ ]:
import os
import json
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import BaseCrossValidator

from azure.identity import DefaultAzureCredential
import adlfs
import mlflow

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

TRAIN_END_YEAR    = 2018   # must match notebook 14

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Mutual information rescue thresholds
MI_TOP_N       = 50   # feature must be in MI top-N to be rescue-eligible
MI_RANK_GAP    = 30   # MI rank must exceed LASSO rank by at least this much

LASSO_N_CS     = 30   # number of regularisation strengths to try
LASSO_MAX_ITER = 5000
RANDOM_STATE   = 42

print(f"Run date       : {RUN_DATE}")
print(f"Train window   : ≤{TRAIN_END_YEAR}")
print(f"Outcomes       : {OUTCOMES}")